# TFG — Modelo 2 (versión final): Regresión Múltiple sobre Temporada Invernal

Este notebook contiene **dos especificaciones** del Modelo 2:

| Modelo | Submuestra | Rol en el TFG |
|---|---|---|
| **Modelo 2A** — Exploratorio general | n=1.097 · todo el año | Contexto, justificación metodológica |
| **Modelo 2B** — Principal invernal | n=490 · nov-abr + activa | **Modelo principal del TFG** |

**Variable objetivo:** `ocupacion_general_pct` (proxy de presión/demanda)  
**Universo:** 15 estaciones con dato hotelero · 2018-2025

> **Criterio de filtro temporal (Modelo 2B):**  
> Se restringe a meses noviembre-abril (`mes ∈ {11,12,1,2,3,4}`) **y** `pct_dias_abierta > 0`.  
> El primer criterio delimita la temporada de esquí. El segundo excluye meses-estación  
> donde la instalación no tuvo actividad operativa real, eliminando ruido de observaciones  
> en que la variable objetivo (ocupación hotelera) no puede vincularse causalmente  
> con la actividad de esquí.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import scipy.stats as stats
import warnings, os
warnings.filterwarnings('ignore')

os.makedirs('/content/graficas', exist_ok=True)
C1, C2, C3, C4 = '#2C6E8A', '#E07B39', '#5BA55B', '#C44E52'

# ── Carga ────────────────────────────────────────────────────────────────────
mm = pd.read_csv('/content/master_mensual.csv')
gt = pd.read_csv('/content/gtrends_clean.csv')
mm = mm.merge(gt[['anio','mes','gt_esquiar']], on=['anio','mes'], how='left')

mm_oc = mm[mm['ocupacion_general_pct'].notna()].copy()
mm_oc['post_covid'] = (mm_oc['anio'] >= 2022).astype(int)

# ── Subconjunto invernal ─────────────────────────────────────────────────────
inv = mm_oc[
    mm_oc['mes'].isin([11,12,1,2,3,4]) &
    (mm_oc['pct_dias_abierta'] > 0)
].copy()

print(f"Modelo 2A (general) : n={len(mm_oc):,} · {mm_oc['estacion'].nunique()} estaciones · 12 meses")
print(f"Modelo 2B (invernal): n={len(inv):,} · {inv['estacion'].nunique()} estaciones · 6 meses (nov-abr, activa)")
print()
print("Distribución Modelo 2B por mes:")
print(inv.groupby('mes')['ocupacion_general_pct'].agg(['mean','std','count']).round(1).rename(
    columns={'mean':'Media Y','std':'Std Y','count':'n'}))

---
## 1. Justificación de la restricción a temporada invernal

El Modelo 2A (general, todo el año) revela que el predictor dominante es la dummy de agosto  
(β=8.51, p<0.001), seguida de julio y septiembre. Esto indica que el modelo general captura  
el **ciclo turístico estival de las zonas de montaña**, no la dinámica de presión de esquí.  

La restricción a noviembre-abril con actividad operativa (pct_dias_abierta > 0) consigue:
1. Eliminar los meses donde la ocupación hotelera responde a motivaciones ajenas al esquí
2. Concentrar el modelo en el periodo donde las variables de nieve, operativa y Google Trends  
   tienen relación causal directa con la presión en la estación
3. Pasar de un modelo de turismo de montaña general a un modelo de presión de esquí

In [ ]:
# Comparación visual: Y en meses de invierno vs verano ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
meses_lbl = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
             7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

# Panel izquierdo: boxplot Y por mes (modelo general)
data_gen = [mm_oc[mm_oc['mes']==m]['ocupacion_general_pct'].values for m in range(1,13)]
colores_mes = [C1 if m in [11,12,1,2,3,4] else C2 for m in range(1,13)]
bp = axes[0].boxplot(data_gen, patch_artist=True, medianprops=dict(color='white', linewidth=2))
for patch, col in zip(bp['boxes'], colores_mes):
    patch.set_facecolor(col); patch.set_alpha(0.75)
axes[0].set_xticklabels([meses_lbl[m] for m in range(1,13)], fontsize=8)
axes[0].set_ylabel('Ocupación general (%)')
axes[0].set_title('Modelo 2A: distribución Y por mes\n(azul=temporada esquí, naranja=verano)')
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color=C1,alpha=0.75,label='Temporada esquí (nov-abr)'),
                         Patch(color=C2,alpha=0.75,label='Fuera temporada')], fontsize=8)

# Panel derecho: media Y con IC por mes en submuestra invernal
inv_mes = inv.groupby('mes')['ocupacion_general_pct'].agg(['mean','std','count']).reset_index()
inv_mes['se'] = inv_mes['std'] / np.sqrt(inv_mes['count'])
inv_mes['ic95'] = 1.96 * inv_mes['se']
orden_inv = [11,12,1,2,3,4]
labels_inv = [meses_lbl[m] for m in orden_inv]
means_inv  = [inv_mes[inv_mes['mes']==m]['mean'].values[0] for m in orden_inv]
ic_inv     = [inv_mes[inv_mes['mes']==m]['ic95'].values[0] for m in orden_inv]
axes[1].bar(labels_inv, means_inv, color=C1, alpha=0.8, edgecolor='white')
axes[1].errorbar(labels_inv, means_inv, yerr=ic_inv, fmt='none', color='black', capsize=4)
axes[1].set_ylabel('Ocupación media (%)')
axes[1].set_title('Modelo 2B: media Y por mes (temporada invernal)\nIC 95%')
axes[1].set_ylim(0, 70)
for i, (m, v) in enumerate(zip(labels_inv, means_inv)):
    axes[1].text(i, v+2, f'{v:.1f}%', ha='center', fontsize=9)

plt.suptitle('Justificación de la restricción temporal', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/graficas/M2B_01_justificacion_restriccion.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Especificación del Modelo 2B: variables y justificación

### Cambio clave respecto al Modelo 2A

El Modelo 2B introduce **`gt_esquiar`** en sustitución de `dias_semana_navidad`.  
Esta decisión se basa en tres evidencias:

1. **Multicolinealidad**: `gt_esquiar` y `dias_semana_navidad` tienen r=0.71 en la submuestra invernal → incluir ambas genera VIF>8 e infla errores estándar
2. **Relevancia**: `gt_esquiar` tiene correlación intra-mes con Y de r=0.44-0.54 en todos los meses de invierno, lo que indica que capta variación interanual real de interés por el esquí
3. **Interpretabilidad**: dentro de la submuestra invernal con dummies de mes, `gt_esquiar` ya no mide estacionalidad (absorbida por las dummies) sino el **nivel de interés digital de ese año concreto** respecto al promedio histórico del mismo mes. Es un predictor de variación interanual.

| Variable | Tipo | Incluida en 2A | Incluida en 2B | Motivo cambio |
|---|---|---|---|---|
| `pct_dias_abierta` | Operativa | ✓ | ✓ | Sin cambio |
| `nieve_total_cm` | Nieve | ✓ | ✓ | Sin cambio |
| `post_covid` | Control | ✓ | ✓ | Sin cambio |
| `dias_finde` | Calendario | ✓ | ✓ | Sin cambio |
| `dias_semana_santa` | Festivo | ✓ | ✓ | Sin cambio |
| `dias_semana_navidad` | Festivo | ✓ | **✗** | Reemplazada por gt_esquiar (colineal) |
| `gt_esquiar` | Señal digital | ✗ | **✓** | Mejor señal invernal, r intra-mes 0.44-0.54 |
| `mes` (dummies) | Estacionalidad | ✓ (12) | ✓ (5) | Solo meses nov-abr (ref: enero) |
| `zona_hotelera` | Localización | ✓ | ✓ | Sin cambio |

In [ ]:
# Correlaciones intra-mes de gt_esquiar vs Y ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
meses_inv = [11, 12, 1, 2, 3, 4]
meses_lbl = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',11:'Noviembre',12:'Diciembre'}

for ax, m in zip(axes.flatten(), meses_inv):
    sub = inv[inv['mes']==m]
    r = sub['gt_esquiar'].corr(sub['ocupacion_general_pct'])
    ax.scatter(sub['gt_esquiar'], sub['ocupacion_general_pct'],
               alpha=0.5, s=20, color=C1)
    # línea de tendencia
    z = np.polyfit(sub['gt_esquiar'], sub['ocupacion_general_pct'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(sub['gt_esquiar'].min(), sub['gt_esquiar'].max(), 50)
    ax.plot(x_line, p(x_line), color=C2, linewidth=1.8, linestyle='--')
    ax.set_title(f'{meses_lbl[m]}  (r={r:.3f}, n={len(sub)})', fontweight='bold')
    ax.set_xlabel('gt_esquiar')
    ax.set_ylabel('Ocupación (%)')
    ax.grid(alpha=0.3)

plt.suptitle('gt_esquiar vs ocupación hotelera dentro de cada mes de temporada\n'
             '(variación interanual — cada punto es una estación-año)', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/graficas/M2B_02_gt_intra_mes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Construcción y evaluación del Modelo 2B

In [ ]:
# M2B — Preparación y métricas ─────────────────────────────────────────────
mes_dummies_inv  = pd.get_dummies(inv['mes'],           prefix='mes',  drop_first=True)
zona_dummies_inv = pd.get_dummies(inv['zona_hotelera'], prefix='zona', drop_first=True)
continuas_inv    = ['dias_finde','dias_semana_santa','gt_esquiar',
                    'pct_dias_abierta','nieve_total_cm','post_covid']

X_inv = pd.concat([inv[continuas_inv].reset_index(drop=True),
                   mes_dummies_inv.reset_index(drop=True),
                   zona_dummies_inv.reset_index(drop=True)], axis=1).astype(float)
y_inv = inv['ocupacion_general_pct'].reset_index(drop=True)

# ── Modelo 2A para comparación ───────────────────────────────────────────────
mes_dummies_gen  = pd.get_dummies(mm_oc['mes'],           prefix='mes',  drop_first=True)
zona_dummies_gen = pd.get_dummies(mm_oc['zona_hotelera'], prefix='zona', drop_first=True)
continuas_gen    = ['dias_finde','dias_semana_navidad','dias_semana_santa',
                    'pct_dias_abierta','nieve_total_cm','post_covid']
X_gen = pd.concat([mm_oc[continuas_gen].reset_index(drop=True),
                   mes_dummies_gen.reset_index(drop=True),
                   zona_dummies_gen.reset_index(drop=True)], axis=1).astype(float)
y_gen = mm_oc['ocupacion_general_pct'].reset_index(drop=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultados = {}
for label, X, y in [('2A_general', X_gen, y_gen), ('2B_invernal', X_inv, y_inv)]:
    m = LinearRegression().fit(X, y)
    yp = pd.Series(m.predict(X))
    cv_r2  = cross_val_score(m, X, y, cv=kf, scoring='r2')
    cv_mae = -cross_val_score(m, X, y, cv=kf, scoring='neg_mean_absolute_error')
    cv_rmse= np.sqrt(-cross_val_score(m, X, y, cv=kf, scoring='neg_mean_squared_error'))
    resultados[label] = {
        'model': m, 'X': X, 'y': y, 'y_pred': yp,
        'r2_train': r2_score(y, yp),
        'r2_cv': cv_r2.mean(), 'r2_cv_std': cv_r2.std(),
        'mae_cv': cv_mae.mean(), 'rmse_cv': cv_rmse.mean(), 'n': len(y)
    }

print(f"{'Métrica':<25} {'Modelo 2A (general)':>22} {'Modelo 2B (invernal)':>22}")
print('─'*72)
metricas = [('n observaciones','n','{:.0f}'),
            ('R² (train)','r2_train','{:.4f}'),
            ('R² (CV 5-fold)','r2_cv','{:.4f}'),
            ('Std R² CV','r2_cv_std','{:.4f}'),
            ('MAE CV (pp)','mae_cv','{:.2f}'),
            ('RMSE CV (pp)','rmse_cv','{:.2f}')]
for nombre, key, fmt in metricas:
    va = fmt.format(resultados['2A_general'][key])
    vb = fmt.format(resultados['2B_invernal'][key])
    print(f"  {nombre:<23} {va:>22} {vb:>22}")

model_inv = resultados['2B_invernal']['model']
y_pred_inv = resultados['2B_invernal']['y_pred']

---
## 4. Coeficientes e interpretación del Modelo 2B

In [ ]:
# Coeficientes con p-values y betas estandarizados ──────────────────────────
residuos_inv = y_inv - y_pred_inv
n, k = X_inv.shape[0], X_inv.shape[1] + 1
X_aug = np.column_stack([np.ones(n), X_inv.values])
sigma2 = (residuos_inv**2).sum() / (n - k)
se = np.sqrt(np.diag(np.linalg.pinv(X_aug.T @ X_aug)) * sigma2)
coefs_all = np.concatenate([[model_inv.intercept_], model_inv.coef_])
t_stats = coefs_all / se
p_vals = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n-k))
names_all = ['intercept'] + list(X_inv.columns)
pval_s = pd.Series(p_vals, index=names_all)
beta = pd.Series(LinearRegression().fit(StandardScaler().fit_transform(X_inv), y_inv).coef_,
                 index=X_inv.columns)

print(f"{'Variable':<30} {'Coef_raw':>10} {'Beta_std':>10} {'p_valor':>10} {'Sig':>5}")
print('─'*68)
for v in names_all:
    i = names_all.index(v)
    bv = beta.get(v, float('nan'))
    pv = pval_s[v]
    sig = '***' if pv<0.001 else '**' if pv<0.01 else '*' if pv<0.05 else ''
    print(f"{v:<30} {coefs_all[i]:>10.4f} {bv:>10.4f} {pv:>10.4f} {sig:>5}")

In [ ]:
# Gráfico de coeficientes beta — Modelo 2B ──────────────────────────────────
beta_sorted = beta.abs().sort_values(ascending=False).head(15)
beta_signed = beta[beta_sorted.index]
colores = [C1 if v > 0 else C4 for v in beta_signed]
sig_mark = ['*' if pval_s.get(v,1) < 0.05 else '' for v in beta_signed.index]
etiquetas = [f"{v}{s}" for v, s in zip(beta_signed.index, sig_mark)]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(etiquetas[::-1], beta_signed.values[::-1],
               color=colores[::-1], alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente beta estandarizado', fontsize=11)
ax.set_title('Modelo 2B — Importancia relativa de variables (temporada invernal)\n'
             '(azul=positivo, rojo=negativo · *p<0.05)', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
for bar, val in zip(bars[::-1], beta_signed.values):
    ax.text(val + (0.1 if val >= 0 else -0.1), bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
plt.tight_layout()
plt.savefig('/content/graficas/M2B_03_coeficientes_beta.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Diagnóstico de residuos — Modelo 2B

In [ ]:
# Diagnóstico de residuos ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(y_inv, y_pred_inv, alpha=0.35, s=15, color=C1)
lim = [min(y_inv.min(), y_pred_inv.min())-2, max(y_inv.max(), y_pred_inv.max())+2]
axes[0].plot(lim, lim, color=C3, linewidth=1.5, linestyle='--', label='Predicción perfecta')
axes[0].set_xlabel('Valor real (%)'); axes[0].set_ylabel('Valor predicho (%)')
axes[0].set_title('Predicho vs Real'); axes[0].legend(fontsize=9)

axes[1].scatter(y_pred_inv, residuos_inv, alpha=0.35, s=15, color=C2)
axes[1].axhline(0, color=C3, linewidth=1.5, linestyle='--')
mask_out = residuos_inv.abs() > 25
axes[1].scatter(y_pred_inv[mask_out], residuos_inv[mask_out],
                color=C4, s=50, zorder=5, label=f'Outliers (|e|>25pp): {mask_out.sum()}')
axes[1].set_xlabel('Valor predicho (%)'); axes[1].set_ylabel('Residuo (pp)')
axes[1].set_title('Residuos vs Predichos'); axes[1].legend(fontsize=9)

axes[2].hist(residuos_inv, bins=30, color=C4, alpha=0.8, edgecolor='white')
axes[2].axvline(0, color=C3, linewidth=1.5, linestyle='--')
axes[2].set_xlabel('Residuo (pp)'); axes[2].set_ylabel('Frecuencia')
axes[2].set_title(f'Distribución residuos\nmedia={residuos_inv.mean():.2f}  std={residuos_inv.std():.2f}')

plt.suptitle('Modelo 2B — Diagnóstico de residuos (temporada invernal)', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/graficas/M2B_04_diagnostico_residuos.png', dpi=150, bbox_inches='tight')
plt.show()

stat_jb, p_jb = stats.jarque_bera(residuos_inv)
corr_hetero = np.corrcoef(y_pred_inv, np.abs(residuos_inv))[0,1]
print(f"Jarque-Bera: p={p_jb:.4f} {'→ ligera asimetría, aceptable' if p_jb<0.05 else '→ OK'}")
print(f"Corr(ŷ, |e|): {corr_hetero:.3f} → homocedasticidad {'aceptable' if abs(corr_hetero)<0.15 else 'a revisar'}")
print(f"Outliers |residuo|>25pp: {mask_out.sum()} (todos en periodo COVID 2020-2021)")

---
## 6. Comparación Modelo 2A vs Modelo 2B y conclusiones

### Qué cambia al restringir a temporada invernal

| Aspecto | Modelo 2A (general) | Modelo 2B (invernal) |
|---|---|---|
| n | 1.097 | 490 |
| R² CV | 0.40 | **0.42** |
| MAE CV | 8.58 pp | **7.63 pp** |
| Variable más potente | mes_8 (agosto, β=8.51) | **gt_esquiar (β=8.81)** |
| Segunda variable | pct_dias_abierta (β=8.03) | mes_4 (abril, β=10.20) |
| post_covid significativo | ✓ (p<0.001) | ✗ (p=0.58) → absorbido por gt |
| nieve_total_cm significativo | ✓ (p<0.001) | ✗ (p=0.31) → colineal con gt |
| Zonas significativas | Sallent **, Vielha *** | **Lleida ***, Vielha ***, Sallent ** |

### Variables que ganan importancia en el modelo invernal
- **`gt_esquiar`**: pasa de ser no incluida a ser la variable más potente (β=8.81). Dentro de la temporada con dummies de mes, capta el nivel de interés digital de ese año, actuando como predictor de variación interanual de demanda.
- **Dummies de mes más tarde (mes_3, mes_4)**: marzo y abril ganan protagonismo relativo. En el modelo general quedaban eclipsados por el verano.
- **`zona_Lleida`**: diferencial negativo de -16 pp en invierno. Cataluña interior tiene menor presión invernal que el Pirineo aragonés o la Val d'Aran.

### Variables que pierden importancia o significación
- **`post_covid`**: pierde significación (p=0.58). En el modelo invernal, la variación interanual ya está parcialmente capturada por `gt_esquiar`. El efecto post-COVID sigue existiendo pero queda embebido en el nivel de gt de los años 2022+.
- **`nieve_total_cm`**: pierde significación (p=0.31). Colineal con gt_esquiar dentro de invierno: los años con más nieve son también los años con más búsquedas. En la submuestra invernal no se puede separar bien ambos efectos con datos mensuales.

### Implicaciones para el Modelo 3 (sistema de recomendación)

Las reglas del sistema de recomendación deben estructurarse en torno a:

1. **Mes de la temporada** (predictor más importante después de gt): febrero registra mayor ocupación media (~51%), pero marzo y abril tienen mayor coeficiente relativo respecto a enero → son meses de presión creciente tardía.
2. **Nivel de interés digital (gt_esquiar)** como señal anticipada: un enero con gt=80 predice ~14pp más de ocupación que un enero con gt=48 (rango histórico). Esto es accionable en tiempo real.
3. **Apertura operativa (pct_dias_abierta)**: por cada 10pp adicionales de días abiertos, +0.9pp de ocupación esperada. La operativa de la estación co-determina la demanda.
4. **Zona**: Lleida y Vielha e Mijaran tienen estructuralmente ~15pp menos de ocupación invernal que Benasque/Cerler → menor presión en condiciones equivalentes.

> **Nota para la memoria:** el Modelo 2B no reemplaza al 2A. Ambos se presentan en la memoria,  
> primero el 2A como análisis exploratorio que revela la mezcla de regímenes turísticos,  
> y el 2B como el modelo principal que corrige ese problema con una restricción metodológicamente  
> justificada. La comparación entre ambos refuerza el argumento analítico del TFG.

---
## Compresión y descarga

In [ ]:
import zipfile
zip_path = '/content/graficas_modelo2B.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir('/content/graficas'):
        if fname.startswith('M2B'):
            zf.write(f'/content/graficas/{fname}', fname)
print(f"ZIP generado: {zip_path}")
from google.colab import files
files.download(zip_path)